In [1]:
!pip install ml_collections

In [1]:
import os
import sys
import shutil
import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F


from tqdm import tqdm, trange

import warnings


from ml_collections import ConfigDict

In [2]:
sys.path.append('..')

In [3]:
from CRT_utils.CRT.core.model1 import Model
# from CRT_utils.CRT.core.config import create_config, save_config
from CRT_utils.CRT.littlehelper import *

# from CRT_utils.CRT.test import test
from CRT_utils.CRT.test_zero_context_zero_target import test

# from CRT_utils.CRT.test_visual_search import test
# from utils.evaluate_uncertainty import evaluate_uncertainty
from CRT_utils.CRT.core.config import create_config, save_config
from CRT_utils.CRT.core.dataset import COCODataset, COCODatasetWithID, COCODatasetMixOR, COCODatasetFullMix
from CRT_utils.CRT.core.model1 import Model
# from CRT_utils.CRT.core.metrics import AccuracyLogger 


# User define Variables (Arguments)

In [4]:
config_dict = ConfigDict()


config_dict['config']                = None
config_dict['outdir']                = '../CRT_utils/CRT_weights_and_config/CRT_COCO_random/Test Model/'
config_dict['checkpoint']            = None # '../CRT_utils/CRT_weights_and_config/CRT_COCO_random/Archive 1/checkpoint_30.tar'

config_dict['annotations']           = '../datasets/COCO18_dset_for_CRT_training/train_metadata.json'
config_dict['imagedir']              = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt'

config_dict['test_annotations']      = '../datasets/COCO18_dset_for_CRT_training/val_metadata.json'
config_dict['test_imagedir']         = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt_test'
config_dict['test_frequency']        = 1

config_dict['epochs']                = 50
config_dict['save_frequency']        = 1
config_dict['print_batch_metrics']   = None

config_dict['batch_size']            = 32 
config_dict['learning_rate']         = None
config_dict['imbalance_reweighting'] = None
config_dict['num_decoder_heads']     = None
config_dict['num_decoder_layers']    = 6
config_dict['uncertainty_gate_type'] = None
config_dict['uncertainty_threshold'] = 0
config_dict['weighted_prediction']   = None

# Setting configure variables

In [5]:
cfg = create_config(config_dict)

In [6]:
dataset = COCODatasetFullMix(
    cfg.annotations, 
    cfg.imagedir, 
    image_size      = (224,224), 
    # _____ ORIGINAL CODE _____
    normalize_means = [0.485, 0.456, 0.406], 
    normalize_stds  = [0.229, 0.224, 0.225],
    # _____ ORIGINAL CODE _____
    
    # _____ MODIFIED VERSION _____
#     normalize_means = [0.5, 0.5, 0.5], 
#     normalize_stds  = [0.5, 5, 0.5]
    # _____ MODIFIED VERSION _____
    
    # _____ ADDED CODE _____
    category_dic_dir = '../datasets/COCO18_dset_for_CRT_training/category_idx_dict.pkl'
    
    # _____ ADDED CODE _____
    
)

dataloader = DataLoader(
    dataset, 
    batch_size  = cfg.batch_size, 
    num_workers = 1, 
    shuffle     = True, 
    pin_memory  = True, 
    drop_last   = True
)


-------------------------------
Annotation Counts
-------------------------------
potted plant               8652
tv                         5805
bottle                    24342
chair                     38491
car                       43867
stop sign                  1983
clock                      6334
cup                       20650
fork                       5479
knife                      7770
bowl                      14358
toilet                     4157
laptop                     4970
mouse                      2262
keyboard                   2855
microwave                  1673
oven                       3334
sink                       5610
Total                    202592
-------------------------------



In [7]:
NUM_CLASSES     = dataset.NUM_CLASSES
cfg.num_classes = NUM_CLASSES

os.makedirs(config_dict.outdir, exist_ok=True)

save_config(cfg, config_dict.outdir)

print(cfg)

annotations: ../datasets/COCO18_dset_for_CRT_training/train_metadata.json
batch_size: 32
checkpoint: null
git: 5b5dc27a31a43c1a58f0121ff3491b0c8215987c
imagedir: ../datasets/COCO18_dset_for_CRT_training/coco18_for_crt
imbalance_reweighting: false
learning_rate: 1.0e-05
num_classes: 18
num_decoder_heads: 8
num_decoder_layers: 6
test_annotations: ../datasets/COCO18_dset_for_CRT_training/val_metadata.json
test_imagedir: ../datasets/COCO18_dset_for_CRT_training/coco18_for_crt_test
uncertainty_gate_type: learned
uncertainty_threshold: 0
weighted_prediction: false



In [8]:
model = Model.from_config(cfg)

In [9]:
assert(model.TARGET_IMAGE_SIZE == model.CONTEXT_IMAGE_SIZE == dataset.image_size), "Image size from the dataset is not compatible with the encoder."

In [10]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps"  # macbook uses metal performance shaders to GPU accelearation
    if torch.backends.mps.is_available()
    else "cpu"
)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay = 0.01)





# _____ ORIGINAL CODE _____

# if cfg.imbalance_reweighting:
#     class_weights = torch.true_divide(dataset.relative_annotation_counts.max(), dataset.relative_annotation_counts)
#     criterion     = nn.CrossEntropyLoss(weight= class_weights.to(device))
# else:
#     criterion = nn.CrossEntropyLoss()
    
# _____ ORIGINAL CODE _____


# _____ MODIFIED CODE _____

criterion = nn.CrossEntropyLoss()

# _____ MODIFIED CODE _____


    
    
    

if cfg.checkpoint is not None:
    
    print("Initializing from checkpoint {}".format(cfg.checkpoint))
    
    checkpoint = torch.load(cfg.checkpoint, map_location="cpu")
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    start_epoch = checkpoint['epoch'] + 1
else:
    print("No checkpoint was passed.")
    
    model.to(device)
    start_epoch = 1

# Tensorboard
writer = SummaryWriter(log_dir=os.path.join(config_dict.outdir, "runs/{date:%Y-%m-%d_%H%M}".format(date=datetime.datetime.now())))

context_images, target_images, bbox, labels = next(iter(dataloader))

writer.add_images("context_image_batch", context_images) # add example context image batch to tensorboard log
writer.add_images("target_image_batch", target_images)   # add example target image batch to tensorboard log

with warnings.catch_warnings(): # add_graph method is known to issue a warning
    warnings.simplefilter("ignore")
    
    # _____ ORIGINAL CODE _____
#     writer.add_graph(model, input_to_model=[context_images.to(device), target_images.to(device), bbox.to(device)]) # add model graph to tensorboard log
    # _____ ORIGINAL CODE _____
    
    
    # _____ MODIFIED VERSION _____
    
    writer.add_graph(
        model, 
        input_to_model=[
            context_images.to(device), 
            target_images.to(device)
        ]
    ) # add model graph to tensorboard log
    
    # _____ MODIFIED VERSION _____
    
    
# _____ ORIGINAL CODE _____
# accuracy_logger_main_branch = AccuracyLogger(dataset.idx2label)
# _____ ORIGINAL CODE _____



# _____ MODIFIED CODE _____
# accuracy_logger_main_branch = AccuracyLogger(num_classes=49)
# _____ MODIFIED CODE _____

No checkpoint was passed.


/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


# Training

In [ ]:
## _____ ORIGINAL CODE _____
# for epoch in tqdm(range(start_epoch, config_dict.epochs + 1), position=0, desc="Epochs", leave=True):

# model.extended_output = True

temperature = 0.1


for epoch in range(start_epoch, config_dict.epochs + 1):

    print(f'Epoch {epoch}/{config_dict.epochs}:')
    model.train() # set train mode
#     accuracy_logger_main_branch.reset() # reset accuracy logger every epoch
    # accuracy_logger_uncertainty_branch.reset()

    for i, (context_images, target_images, bbox, labels_cpu) in enumerate(tqdm(dataloader, position=0, desc="Batches", leave=True)):
        
        context_images = context_images.to(device)
        target_images  = target_images.to(device)
        
        bbox   = bbox.to(device)
        labels = labels_cpu.to(device) # keep a copy of labels on cpu to avoid unnecessary transfer back to cpu later

        # output_uncertainty_branch , output_main_branch, output_weighted, uncertainty = model(context_images, target_images, bbox)
        
        
        # _____ ORIGINAL CODE _____
#         output_main_branch = model(context_images, target_images, bbox)
        # _____ ORIGINAL CODE _____
    
    
    
        # _____ MODIFIED VERSION _____
        output_main_branch, attention_map = model(context_images, target_images)
        
        attention_map = attention_map[:, cfg.num_decoder_layers - 1, 0]
        
        
        # _____ MODIFIED VERSION _____
        

        # backpropagation through both branches
        optimizer.zero_grad(set_to_none=True)

        # if cfg.uncertainty_gate_type == "learned" or cfg.uncertainty_gate_type == "learned_metric":
        #     loss_uncertainty_estimator = criterion(output_weighted, labels)
        #     loss_uncertainty_estimator.backward(retain_graph=True)    

        # loss_uncertainty_branch = criterion(output_uncertainty_branch, labels)
        # loss_uncertainty_branch.backward(retain_graph=True)

#         print(attention_map.shape)
#         print(bbox2token(model, bbox).shape)
        
        # _____ MODIFIED CODE _____
        
        
        loss_main_branch = criterion(output_main_branch, labels)
        
#         loss_main_branch = criterion(
#             torch.log(attention_map + 1e-8), 
#             bbox2token(model, bbox)
#         )
        # _____ MODIFIED CODE _____
        
        

        
        loss_main_branch.backward()

        optimizer.step()
        
        # log metrics
        # _, predictions_uncertainty_branch = torch.max(output_uncertainty_branch.detach().to("cpu"), 1) # choose idx with maximum score as prediction
        # batch_accuracy_uncertainty_branch = sum(predictions_uncertainty_branch == labels_cpu) / cfg.batch_size
        # batch_loss_uncertainty_branch = loss_uncertainty_branch.item()
        # writer.add_scalar("Batch Accuracy Uncertainty Branch/train", batch_accuracy_uncertainty_branch, i + (epoch - 1) * len(dataloader))
        # writer.add_scalar("Batch Loss Uncertainty Branch/train", batch_loss_uncertainty_branch, i + (epoch - 1) * len(dataloader))
        # accuracy_logger_uncertainty_branch.update(predictions_uncertainty_branch, labels_cpu)

        
        
#         _, predictions_main_branch = torch.max(output_main_branch.detach().to("cpu"), 1) # choose idx with maximum score as prediction
        
        predictions_main_branch = torch.argmax(output_main_branch.detach().to("cpu"), 1)
        
        batch_accuracy_main_branch = sum(predictions_main_branch == labels.to('cpu')) / cfg.batch_size
        batch_loss_main_branch     = loss_main_branch.item()
        
        writer.add_scalar("Batch Accuracy Main Branch/train", batch_accuracy_main_branch, i + (epoch - 1) * len(dataloader))
        
        writer.add_scalar("Batch Loss Main Branch/train", batch_loss_main_branch, i + (epoch - 1) * len(dataloader))
#         accuracy_logger_main_branch.update(predictions_main_branch, bbox2token(model, bbox).to('cpu'))

        # writer.add_scalar("Batch Uncertainty/train", torch.mean(uncertainty), i + (epoch - 1) * len(dataloader))

        if config_dict.print_batch_metrics:
            print("\t Epoch {}, Batch {}: \t Loss: {} \t Accuracy: {}".format(epoch, i, batch_loss_main_branch, batch_accuracy_main_branch))


    # log metrics
#     writer.add_scalar("Total Accuracy Main Branch/train", accuracy_logger_main_branch.accuracy(), epoch * len(dataloader))
#     writer.add_scalar("Total Accuracy Uncertainty Branch/train", accuracy_logger_uncertainty_branch.accuracy(), epoch * len(dataloader))

#     print("\nEpoch {}, Train Accuracy: {}".format(epoch, accuracy_logger_main_branch.accuracy()))
#     print("{0:20} {1:10}".format("Class", "Accuracy")) # header
    
    
#     for name, acc in accuracy_logger_main_branch.named_class_accuarcies().items():
#         writer.add_scalar("Class Accuracies Main Branch/train/{}".format(name), acc, epoch * len(dataloader))
#         print("{0:20} {1:10.4f}".format(name, acc))

    # for name, acc in accuracy_logger_uncertainty_branch.named_class_accuarcies().items():
    #     writer.add_scalar("Class Accuracies Uncertainty Branch/train/{}".format(name), acc, epoch * len(dataloader))

    # save checkpoint and training accuracies
    if epoch % config_dict.save_frequency == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()}, config_dict.outdir + "/checkpoint_{}.tar".format(epoch))
        print("Checkpoint saved.")

#         accuracy_logger_main_branch.save(config_dict.outdir, name="train_accuracies_epoch_{}".format(epoch))
        # accuracy_logger_uncertainty_branch.save(args.outdir, name="train_accuracies_uncertainty_branch_epoch_{}".format(epoch))
    
    # evaluation on test data
    
    
    
    
    
    
    
    
    
    if cfg.test_annotations is not None and cfg.test_imagedir is not None and epoch % config_dict.test_frequency == 0:
        print("Starting evaluation on test data.")
        test_accuracy = test(
            model, 
            cfg.test_annotations, 
            cfg.test_imagedir, 
#             '../datasets/COCO18_dset_for_CRT_training/category_idx_dict.pkl',
            outdir = config_dict.outdir,
            epoch  = epoch,
        )
        
        
        
        
        
        
        
        
        
        
        
        

        writer.add_scalar("Total Accuracy/test", test_accuracy.accuracy(), epoch * len(dataloader))
        
        for name, acc in test_accuracy.named_class_accuarcies().items():
            writer.add_scalar("Class Accuracies/test/{}".format(name), acc, epoch * len(dataloader))

        # print("Starting uncertainty evaluation.")
        # test_uncertainty_log = evaluate_uncertainty(model, cfg.test_annotations, cfg.test_imagedir)
        # writer.add_figure("Uncertainty Threshold Curve", test_uncertainty_log.plot_accuracy_vs_threshold(), epoch * len(dataloader))

        # if (args.epochs - epoch) / args.test_frequency < 1: # last evaluation
        #     writer.add_hparams({"learning_rate": cfg.learning_rate, "num_decoder_layers": cfg.num_decoder_layers, "num_decoder_heads": cfg.num_decoder_heads,
        #                         "uncertainty_gate_type": cfg.uncertainty_gate_type, "uncertainty_threshold": cfg.uncertainty_threshold, "imbalance_reweighting": str(cfg.imbalance_reweighting)},
        #                         metric_dict={"hparam/accuracy": test_accuracy.accuracy()})
        
writer.close()

Epoch 1/50:


Batches:   0%|                                                                                                                          | 0/6331 [00:00<?, ?it/s]/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6331/6331 [1:13:10<00:00,  1.44it/s]


Checkpoint saved.
Starting evaluation on test data.
-------------------------------
Annotation Counts
-------------------------------
potted plant                343
tv                          288
bottle                     1025
chair                      1791
car                        1932
stop sign                    75
clock                       267
cup                         899
fork                        215
knife                       326
bowl                        626
toilet                      179
laptop                      231
mouse                       106
keyboard                    153
microwave                    55
oven                        143
sink                        225
Total                      8879
-------------------------------



Test Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 8879/8879 [09:03<00:00, 16.33it/s]



Total Test Accuracy: 0.3494512140750885
Class                Accuracy  
car                      0.8856
stop sign                0.0400
bottle                   0.4585
cup                      0.1123
fork                     0.2744
knife                    0.1411
bowl                     0.4137
chair                    0.8353
potted plant             0.1691
toilet                   0.6648
tv                       0.5625
laptop                   0.2944
mouse                    0.1698
keyboard                 0.1830
microwave                0.0182
oven                     0.3636
sink                     0.3067
clock                    0.3970
Epoch 2/50:


...

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6331/6331 [1:26:06<00:00,  1.23it/s]


Checkpoint saved.
Starting evaluation on test data.
-------------------------------
Annotation Counts
-------------------------------
potted plant                343
tv                          288
bottle                     1025
chair                      1791
car                        1932
stop sign                    75
clock                       267
cup                         899
fork                        215
knife                       326
bowl                        626
toilet                      179
laptop                      231
mouse                       106
keyboard                    153
microwave                    55
oven                        143
sink                        225
Total                      8879
-------------------------------



Test Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 8879/8879 [10:31<00:00, 14.06it/s]



Total Test Accuracy: 0.5290325880050659
Class                Accuracy  
car                      0.9042
stop sign                0.4133
bottle                   0.5678
cup                      0.4194
fork                     0.3023
knife                    0.2209
bowl                     0.5176
chair                    0.8146
potted plant             0.3440
toilet                   0.6816
tv                       0.5799
laptop                   0.4069
mouse                    0.7264
keyboard                 0.5621
microwave                0.3818
oven                     0.4545
sink                     0.6222
clock                    0.6030
Epoch 16/50:


Batches:   0%|                                                                                                                          | 0/6331 [00:00<?, ?it/s]

In [ ]:
test_accuracy = test(
    model, 
    cfg.test_annotations, 
    cfg.test_imagedir, 
#             '../datasets/COCO18_dset_for_CRT_training/category_idx_dict.pkl',
    outdir = config_dict.outdir,
    epoch  = epoch,
)

In [ ]:
len(dataset)

In [ ]:
predictions_main_branch

In [16]:
bbox2token(model, bbox).to('cpu')

tensor([39, 28, 15, 14, 32, 18, 36, 33, 37, 12, 33, 31, 25,  8, 29, 41, 16, 28,
        16, 10, 23, 30, 31, 34,  1, 17,  0, 25, 38, 15, 12, 38])

# Forwarding process

In [21]:
target_encoding = model.target_encoder(target_images)

In [22]:
target_encoding.shape

torch.Size([32, 1664, 7, 7])

In [23]:
context_encoding = model.context_encoder(context_images)

In [24]:
context_encoding.shape

torch.Size([32, 1664, 7, 7])

In [25]:
context_encoding1, target_encoding1 = model.tokenizer(context_encoding, target_encoding)

In [26]:
context_encoding1.shape

torch.Size([49, 32, 1664])

In [27]:
target_encoding1.shape

torch.Size([1, 32, 1664])

In [28]:
context_encoding2, target_encoding2 = model.positional_encoding(context_encoding1, target_encoding1)

In [29]:
context_encoding2.shape

torch.Size([49, 32, 1664])

In [30]:
target_encoding2.shape

torch.Size([1, 32, 1664])

In [32]:
target_encoding3, attention_map = model.decoder(target_encoding2, context_encoding2)

In [33]:
target_encoding3.shape

torch.Size([1, 32, 1664])

In [34]:
attention_map.shape

torch.Size([32, 6, 1, 49])

In [28]:
torch.tensor([
    [[1,2]], [[1,2]], [[1,2]]
]).squeeze(1).shape

torch.Size([3, 2])

In [32]:
torch.tensor([
    [[1,2]], [[1,2]], [[1,2]]
])[:, 0].shape

torch.Size([3, 2])

In [15]:
bbox2token(model, bbox)

tensor([25, 22, 30, 39, 17, 20, 38, 41, 17, 36, 24, 40, 10,  7, 23, 24, 18, 16,
        28, 21, 22, 28, 26, 32, 24, 36,  5,  4, 34, 41, 12, 25],
       device='mps:0')

In [13]:
model.NUM_CONTEXT_TOKENS

49

In [ ]:
train.py
import os
import warnings
import argparse
import datetime
import pathlib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from tqdm import tqdm, trange

from test import test
from utils.evaluate_uncertainty import evaluate_uncertainty
from core.config import create_config, save_config
from core.dataset import COCODataset, COCODatasetWithID, COCODatasetGeneral
from core.model import Model
from core.metrics import AccuracyLogger


## Initialization
#
    
parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, help="Path to config file. If additional commandline options are provided, they are used to modify the specifications in the config file.")
parser.add_argument("--outdir", type=str, default="output/{date:%Y-%m-%d_%H%M}".format(date=datetime.datetime.now()), help="Path to output folder (will be created if it does not exist).")
parser.add_argument("--checkpoint", type=str, help="Path to model checkpoint from which to continue training.")
parser.add_argument("--annotations", type=str, help="Path to COCO-style annotations file.")
parser.add_argument("--imagedir", type=str, help="Path to images folder w.r.t. which filenames are specified in the annotations.")

parser.add_argument("--test_annotations", type=str, help="Path to COCO-style annotations file for model evaluation.")
parser.add_argument("--test_imagedir", type=str, help="Path to images folder w.r.t. which filenames are specified in the annotations for model evaluation.")
parser.add_argument("--test_frequency", type=int, default=1, help="Evaluate model on test data every __ epochs.")

parser.add_argument("--epochs", type=int, default=1, help="Number of epochs to train.")
parser.add_argument("--save_frequency", type=int, default=1, help="Save model checkpoint every __ epochs.")
parser.add_argument("--print_batch_metrics", action='store_true', default=False, help="Set to print metrics for every batch.")

parser.add_argument("--batch_size", type=int, help="Batchsize to use for training.")
parser.add_argument("--learning_rate", type=float, help="Learning rate to use for training.")
parser.add_argument("--imbalance_reweighting", action='store_true', help="Reweight samples in proportion to the number of samples per class.")
parser.add_argument("--num_decoder_heads", type=int, help="Number of decoder heads.")
parser.add_argument("--num_decoder_layers", type=int, help="Number of decoder layers.")
parser.add_argument("--uncertainty_gate_type", type=str, help="Uncertainty gating mechanism to use. Can be one of: 'entropy', 'relative_softmax_distance', 'learned', 'learned_metric'.")
parser.add_argument("--uncertainty_threshold", type=float, help="Uncertainty threshold for the uncertainty gating module. Note that training does not depend on the threshold, the model can still be used with different thresholds later.")
parser.add_argument("--weighted_prediction", action='store_true', default=None, help="If enabled, the model returns an uncertainty-weighted prediction if the uncertainty_gate prediction exceeds the uncertainty threshold.")
args = parser.parse_args()

# Create output directory
pathlib.Path(args.outdir).mkdir(exist_ok=True, parents=True)

# Load config or create a new one
cfg = create_config(args)

dataset = COCODatasetGeneral(cfg.annotations, cfg.imagedir, image_size =(224,224), normalize_means=[0.485, 0.456, 0.406], normalize_stds=[0.229, 0.224, 0.225])
dataloader = DataLoader(dataset, batch_size=cfg.batch_size, num_workers=4, shuffle=True, pin_memory=True, drop_last=True)

NUM_CLASSES = dataset.NUM_CLASSES
cfg.num_classes = NUM_CLASSES
save_config(cfg, args.outdir)
print(cfg)

model = Model.from_config(cfg)

assert(model.TARGET_IMAGE_SIZE == model.CONTEXT_IMAGE_SIZE == dataset.image_size), "Image size from the dataset is not compatible with the encoder."

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)

if cfg.imbalance_reweighting:
    class_weights = torch.true_divide(dataset.relative_annotation_counts.max(), dataset.relative_annotation_counts)
    criterion = nn.CrossEntropyLoss(weight= class_weights.to(device))
else:
    criterion = nn.CrossEntropyLoss()

if cfg.checkpoint is not None:
    print("Initializing from checkpoint {}".format(cfg.checkpoint))
    checkpoint = torch.load(cfg.checkpoint, map_location="cpu")
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
else:
    print("No checkpoint was passed.")
    model.to(device)
    start_epoch = 1

# Tensorboard
writer = SummaryWriter(log_dir=os.path.join(args.outdir, "runs/{date:%Y-%m-%d_%H%M}".format(date=datetime.datetime.now())))
context_images, target_images, bbox, labels = iter(dataloader).next()
writer.add_images("context_image_batch", context_images) # add example context image batch to tensorboard log
writer.add_images("target_image_batch", target_images) # add example target image batch to tensorboard log
with warnings.catch_warnings(): # add_graph method is known to issue a warning
    warnings.simplefilter("ignore")
    writer.add_graph(model, input_to_model=[context_images.to(device), target_images.to(device), bbox.to(device)]) # add model graph to tensorboard log

accuracy_logger_main_branch = AccuracyLogger(dataset.idx2label)
# accuracy_logger_uncertainty_branch = AccuracyLogger(dataset.idx2label)


## Training
#

for epoch in tqdm(range(start_epoch, args.epochs + 1), position=0, desc="Epochs", leave=True):

    model.train() # set train mode
    accuracy_logger_main_branch.reset() # reset accuracy logger every epoch
    # accuracy_logger_uncertainty_branch.reset()

    for i, (context_images, target_images, bbox, labels_cpu) in enumerate(tqdm(dataloader, position=1, desc="Batches", leave=True)):
        context_images = context_images.to(device)
        target_images = target_images.to(device)
        bbox = bbox.to(device)
        labels = labels_cpu.to(device) # keep a copy of labels on cpu to avoid unnecessary transfer back to cpu later

        # output_uncertainty_branch , output_main_branch, output_weighted, uncertainty = model(context_images, target_images, bbox)
        output_main_branch = model(context_images, target_images, bbox)

        # backpropagation through both branches
        optimizer.zero_grad(set_to_none=True)

        # if cfg.uncertainty_gate_type == "learned" or cfg.uncertainty_gate_type == "learned_metric":
        #     loss_uncertainty_estimator = criterion(output_weighted, labels)
        #     loss_uncertainty_estimator.backward(retain_graph=True)    

        # loss_uncertainty_branch = criterion(output_uncertainty_branch, labels)
        # loss_uncertainty_branch.backward(retain_graph=True)

        loss_main_branch = criterion(output_main_branch, labels)
        loss_main_branch.backward()

        optimizer.step()
        
        # log metrics
        # _, predictions_uncertainty_branch = torch.max(output_uncertainty_branch.detach().to("cpu"), 1) # choose idx with maximum score as prediction
        # batch_accuracy_uncertainty_branch = sum(predictions_uncertainty_branch == labels_cpu) / cfg.batch_size
        # batch_loss_uncertainty_branch = loss_uncertainty_branch.item()
        # writer.add_scalar("Batch Accuracy Uncertainty Branch/train", batch_accuracy_uncertainty_branch, i + (epoch - 1) * len(dataloader))
        # writer.add_scalar("Batch Loss Uncertainty Branch/train", batch_loss_uncertainty_branch, i + (epoch - 1) * len(dataloader))
        # accuracy_logger_uncertainty_branch.update(predictions_uncertainty_branch, labels_cpu)

        _, predictions_main_branch = torch.max(output_main_branch.detach().to("cpu"), 1) # choose idx with maximum score as prediction
        batch_accuracy_main_branch = sum(predictions_main_branch == labels_cpu) / cfg.batch_size
        batch_loss_main_branch = loss_main_branch.item()
        writer.add_scalar("Batch Accuracy Main Branch/train", batch_accuracy_main_branch, i + (epoch - 1) * len(dataloader))
        writer.add_scalar("Batch Loss Main Branch/train", batch_loss_main_branch, i + (epoch - 1) * len(dataloader))
        accuracy_logger_main_branch.update(predictions_main_branch, labels_cpu)

        # writer.add_scalar("Batch Uncertainty/train", torch.mean(uncertainty), i + (epoch - 1) * len(dataloader))

        if args.print_batch_metrics:
            print("\t Epoch {}, Batch {}: \t Loss: {} \t Accuracy: {}".format(epoch, i, batch_loss_main_branch, batch_accuracy_main_branch))


    # log metrics
    writer.add_scalar("Total Accuracy Main Branch/train", accuracy_logger_main_branch.accuracy(), epoch * len(dataloader))
    # writer.add_scalar("Total Accuracy Uncertainty Branch/train", accuracy_logger_uncertainty_branch.accuracy(), epoch * len(dataloader))

    print("\nEpoch {}, Train Accuracy: {}".format(epoch, accuracy_logger_main_branch.accuracy()))
    print("{0:20} {1:10}".format("Class", "Accuracy")) # header
    for name, acc in accuracy_logger_main_branch.named_class_accuarcies().items():
        writer.add_scalar("Class Accuracies Main Branch/train/{}".format(name), acc, epoch * len(dataloader))
        print("{0:20} {1:10.4f}".format(name, acc))

    # for name, acc in accuracy_logger_uncertainty_branch.named_class_accuarcies().items():
    #     writer.add_scalar("Class Accuracies Uncertainty Branch/train/{}".format(name), acc, epoch * len(dataloader))

    # save checkpoint and training accuracies
    if epoch % args.save_frequency == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()}, args.outdir + "/checkpoint_{}.tar".format(epoch))
        print("Checkpoint saved.")

        accuracy_logger_main_branch.save(args.outdir, name="train_accuracies_epoch_{}".format(epoch))
        # accuracy_logger_uncertainty_branch.save(args.outdir, name="train_accuracies_uncertainty_branch_epoch_{}".format(epoch))
    
    # evaluation on test data
    if cfg.test_annotations is not None and cfg.test_imagedir is not None and epoch % args.test_frequency == 0:
        print("Starting evaluation on test data.")
        test_accuracy = test(model, cfg.test_annotations, cfg.test_imagedir, outdir=args.outdir, epoch=epoch)

        writer.add_scalar("Total Accuracy/test", test_accuracy.accuracy(), epoch * len(dataloader))
        for name, acc in test_accuracy.named_class_accuarcies().items():
            writer.add_scalar("Class Accuracies/test/{}".format(name), acc, epoch * len(dataloader))

        # print("Starting uncertainty evaluation.")
        # test_uncertainty_log = evaluate_uncertainty(model, cfg.test_annotations, cfg.test_imagedir)
        # writer.add_figure("Uncertainty Threshold Curve", test_uncertainty_log.plot_accuracy_vs_threshold(), epoch * len(dataloader))

        # if (args.epochs - epoch) / args.test_frequency < 1: # last evaluation
        #     writer.add_hparams({"learning_rate": cfg.learning_rate, "num_decoder_layers": cfg.num_decoder_layers, "num_decoder_heads": cfg.num_decoder_heads,
        #                         "uncertainty_gate_type": cfg.uncertainty_gate_type, "uncertainty_threshold": cfg.uncertainty_threshold, "imbalance_reweighting": str(cfg.imbalance_reweighting)},
        #                         metric_dict={"hparam/accuracy": test_accuracy.accuracy()})
        
writer.close()